In [1]:
from rdflib import Graph

from schema_definition import *
from execute_sparql import execute_sparql
from graphdb_engine import query_graphdb


In [ ]:
# 1. Initialize the Graph
g = Graph()

# 2. Parse the .ttl file
# Replace "data.ttl" with the path to your actual file
g.parse("output/mtrKG.ttl", format="turtle")

# 3. Verify it loaded correctly by printing the number of triples
print(f"The graph contains {len(g)} triples.")

# Query 01

<font size="4">
The following query shows the number of entities and edges each data source contributed to the graph. This is a critical sanity check to ensure that all your data sources were ingested correctly and are represented in the graph as expected.
</font>

In [13]:
df_results = query_graphdb("../analysis/001_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/001_query.rq
SPARQL query being executed:
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>
PREFIX prov: <http://www.w3.org/ns/prov#>

SELECT ?dataset_source (COUNT(DISTINCT ?entity) AS ?contribution_count)
WHERE {
  {
    # 1. rQTL
    { ?entity rdf:type mtr:VariantToMetaboliteRatioAssociation }
    UNION
    { ?entity rdf:type mtr:CausalAssessment }
    BIND("rQTL" AS ?dataset_source)
  }
  UNION
  {
    # 2. GWAS Catalog
    ?entity mtr:ci_lower ?any1 .
    BIND("GWAS Catalog" AS ?dataset_source)
  }
  UNION
  {
    # 3. OpenTargets
    { ?entity mtr:ot_genetics_score ?any2 }
    UNION
    { ?entity rdf:type mtr:TargetSafetyLiabilityAssociation }
    BIND("OpenTargets" AS ?dataset_source)
  }
  UNION
  {
    # 4. HMDB
    { ?entity rdf:type mtr:MetaboliteLocationAssociation }
    UNION
    { ?entity mtr:hmdb_id ?any3 }
    BIN

,dataset_source,contribution_count
0,rQTL,231268
1,Reactome,15514
2,GWAS Catalog,5229
3,STRING,3180
4,OpenTargets,751
5,Rhea,725
6,HMDB,284
7,Ensembl,62


# Query 02

<font size="4"> "What are the most statistically significant genetic associations in the GWAS Catalog currently stored in our graph, and what are their specific effect sizes, risk alleles, and original publication IDs?"
</font>

In [2]:
df_results = query_graphdb("../analysis/002_query.rq", repo_name="mtrKG")

# Display the results
df_results

Reading SPARQL query from file: ../analysis/002_query.rq
SPARQL query being executed:
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>
PREFIX prov: <http://www.w3.org/ns/prov#>

SELECT
  ?snp_id
  (MIN(?p_value) AS ?strongest_gwas_p_value)
  (GROUP_CONCAT(DISTINCT ?trait_name; separator=" | ") AS ?associated_clinical_traits)
  (GROUP_CONCAT(DISTINCT ?ratio_name; separator=" | ") AS ?driven_metabolic_ratios)
WHERE {
  # 1. GWAS Catalog Associations (Clinical)
  ?gwas_assoc a biolink:Association ;
              prov:wasGeneratedBy "EBI_GWAS_REST_API_v2" ;
              biolink:has_subject ?snp ;
              biolink:has_object ?trait ;
              mtr:p_value ?p_value .

  ?trait rdfs:label ?trait_name .

  # Strict significance filter
  FILTER(?p_value < 0.00000005)

  # 2. Local rQTL Associations (Metabolic)
  # We require the SNP to also be linked to at least one ratio in 

,snp_id,strongest_gwas_p_value,associated_clinical_traits,driven_metabolic_ratios
0,rs102275,4e-264,coronary artery calcification | lysophosphatid...,"1,2-dilinoleoyl-gpc (18:2/18:2) to 1-lignocero..."
1,rs1440581,4e-264,coronary artery calcification | metabolite mea...,tiglylcarnitine (c5:1-dc) to 4-methyl-2-oxopen...
2,rs4788815,4e-264,coronary artery calcification | metabolite mea...,gamma-glutamylleucine to gamma-glutamyltyrosine
3,rs62129966,2e-108,"vitamin D amount | androstenediol (3alpha, 17a...",pregnanediol-3-glucuronide to 5alpha-pregnan-3...
4,rs77697233,2e-108,vitamin D amount,"5alpha-androstan-3alpha,17alpha-diol monosulfa..."
...,...,...,...,...
594,rs10212439,3.0000000000000004e-08,level of phosphatidylinositol,1-linoleoyl-gpe (18:2)* to 1-oleoyl-gpe (18:1)
595,rs34613987,3.0000000000000004e-08,staphylococcus seropositivity,x-21312 to x-19141 | 21-hydroxypregnenolone di...
596,rs115430557,3.0000000000000004e-08,gut microbiome measurement | breastfeeding dur...,"7-methylxanthine to 3,7-dimethylurate"
597,rs7572565,3.0000000000000004e-08,gut microbiome measurement | breastfeeding dur...,"n2,n5-diacetylornithine to n-delta-acetylornit..."


c# Query 03

<font size="4"> Find me a severe clinical disease from the GWAS Catalog that is driven by a specific genetic variant. Then, show me the metabolic ratio disrupted by that variant, the causal enzyme responsible for the disruption, and finally, check if there is an FDA-approved drug (Phase 3 or 4) that already targets that enzyme."
</font>

In [3]:
df_results = query_graphdb("../analysis/003_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/003_query.rq
SPARQL query being executed:
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>

SELECT DISTINCT ?disease_name ?ratio_name ?causal_gene ?drug_name ?max_phase
WHERE {
  # ========================================================
  # 1. THE PHARMACOLOGY (Open Targets)
  # Find advanced clinical drugs and their target genes
  # ========================================================
  ?drug a biolink:Drug ;
        biolink:targets ?gene ;
        rdfs:label ?drug_name ;
        mtr:max_clinical_phase ?max_phase .

  FILTER(?max_phase >= 3.0)

  ?gene a mtr:Gene ;
        rdfs:label ?causal_gene .

  # ========================================================
  # 2. THE METABOLIC MECHANISM (Local MR Data)
  # Find metabolite ratios causally controlled by these targets
  # ========================================================

,disease_name,ratio_name,causal_gene,drug_name,max_phase
0,Alzheimer disease,1-(1-enyl-palmitoyl)-2-oleoyl-gpc (p-16:0/18:1...,GRIN3B,FELBAMATE,4
1,Alzheimer disease,1-(1-enyl-palmitoyl)-2-palmitoyl-gpc (p-16:0/1...,GRIN3B,FELBAMATE,4
2,Alzheimer disease,1-(1-enyl-palmitoyl)-2-oleoyl-gpc (p-16:0/18:1...,GRIN3B,FELBAMATE,4
3,Alzheimer disease,1-(1-enyl-palmitoyl)-2-palmitoyl-gpc (p-16:0/1...,GRIN3B,FELBAMATE,4
4,Alzheimer disease,1-(1-enyl-palmitoyl)-2-oleoyl-gpc (p-16:0/18:1...,GRIN3B,ORPHENADRINE CITRATE,4
...,...,...,...,...,...
2159,whole body water mass,"n6,n6,n6-trimethyllysine to argininate*",E2F4,EDIFOLIGIDE SODIUM,3
2160,whole body water mass,"n6,n6,n6-trimethyllysine to deoxycarnitine",E2F4,EDIFOLIGIDE SODIUM,3
2161,whole body water mass,"n6,n6,n6-trimethyllysine to argininate*",E2F4,EDIFOLIGIDE SODIUM,3
2162,whole body water mass,"n6,n6,n6-trimethyllysine to deoxycarnitine",E2F4,EDIFOLIGIDE SODIUM,3.0


# Query 04

<font size="4"> We found a gene that causally drives a metabolic imbalance linked to a disease. However, what if that specific gene is considered 'undruggable' (no known drugs)? Can the Knowledge Graph find an FDA-approved drug that targets a different protein, as long as that protein physically binds to our undruggable causal gene?"
</font>

In [4]:
df_results = query_graphdb("../analysis/004_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/004_query.rq
SPARQL query being executed:
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>

SELECT DISTINCT
  ?ratio_name
  ?causal_gene_name
  ?interacting_gene_name
  ?interaction_score
  ?drug_name
  ?clinical_phase
WHERE {
  # ========================================================
  # 1. THE BIOMARKER (Local MR Data)
  # Find a gene that causally drives a metabolic ratio
  # ========================================================
  ?causal_assessment a mtr:CausalAssessment ;
                     mtr:has_outcome ?ratio ;
                     mtr:has_exposure ?causal_gene .

  ?ratio a mtr:MetaboliteRatio ; rdfs:label ?ratio_name .
  ?causal_gene a mtr:Gene ; rdfs:label ?causal_gene_name .

  # ========================================================
  # 2. THE NETWORK BRIDGE (STRING Database)
  # Find proteins that physically i

,ratio_name,causal_gene_name,interacting_gene_name,interaction_score,drug_name,clinical_phase
0,1-palmitoleoyl-gpc (16:1)* to stearoylcarnitin...,NDUFB8,NDUFA7,0.999,METFORMIN,4
1,"sphingomyelin (d18:1/18:1, d18:2/18:0) to sphi...",NDUFB8,NDUFA7,0.999,METFORMIN,4
2,1-palmitoleoyl-gpc (16:1)* to 1-myristoyl-2-pa...,NDUFB8,NDUFA7,0.999,METFORMIN,4
3,1-palmitoleoyl-gpc (16:1)* to 1-palmitoyl-gpc ...,NDUFB8,NDUFA7,0.999,METFORMIN,4
4,1-stearoyl-gpg (18:0) to 2-palmitoleoyl-gpc (1...,NDUFB8,NDUFA7,0.999,METFORMIN,4
5,2-palmitoleoyl-gpc (16:1)* to stearoylcarnitin...,NDUFB8,NDUFA7,0.999,METFORMIN,4
6,"sphingomyelin (d18:1/18:1, d18:2/18:0) to palm...",NDUFB8,NDUFA7,0.999,METFORMIN,4
7,"sphingomyelin (d18:1/18:1, d18:2/18:0) to sphi...",NDUFB8,NDUFA7,0.999,METFORMIN,4
8,1-palmitoyl-2-palmitoleoyl-gpc (16:0/16:1)* to...,NDUFB8,NDUFA7,0.999,METFORMIN,4
9,"sphingomyelin (d18:2/24:1, d18:1/24:2)* to sph...",NDUFB8,NDUFA7,0.999,METFORMIN,4


# Query 05

<font size="4">
This shows the regulatory elements that have been added by the ensembl dataset
</font>

In [5]:
df_results = query_graphdb("../analysis/005_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/005_query.rq
SPARQL query being executed:
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>

SELECT  ?snp_id ?regulatory_element
WHERE {
  # 1. Find SNPs associated with a clinical trait in GWAS

  # 2. Find ENCODE Epigenetic features that the SNP physically overlaps
  ?snp biolink:overlaps ?reg_node .
  ?reg_node rdfs:label ?regulatory_element .

  BIND(REPLACE(STR(?snp), "^.*/", "") AS ?snp_id)
}
ORDER BY ?disease_name
Sending query to GraphDB (mtrKG)...
Success! Retrieved 124 rows.
Saved results to CSV: ../analysis/005_query.csv


,snp_id,regulatory_element
0,rs174560,enhancer (ENSR11_C85LJ)
1,rs174560,enhancer (ENSR11_C85LJ)
2,rs174564,enhancer (ENSR11_9WJJ2)
3,rs174564,enhancer (ENSR11_9WJJ2)
4,rs174695,enhancer (ENSR22_C3ZGS)
...,...,...
119,rs7705826,enhancer (ENSR5_93XJT9)
120,rs9395857,enhancer (ENSR6_9R6C4)
121,rs9395857,enhancer (ENSR6_9R6C4)
122,rs999992,enhancer (ENSR1_B356L8)


# Query 06

<font size="4">
"If my MR data says Gene X causes a change in Metabolite Y, do they actually share a known biological pathway in Reactome?
</font>


In [6]:
df_results = query_graphdb("../analysis/006_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/006_query.rq
SPARQL query being executed:
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>

SELECT DISTINCT ?gene_name ?metabolite_name ?pathway_name
WHERE {
  # 1. Find a causal relationship from your local MR data
  ?causal_test a mtr:CausalAssessment ;
               mtr:has_exposure ?gene ;
               mtr:has_outcome ?ratio .

  # 2. Get the numerator metabolite of that ratio
  ?ratio mtr:has_numerator ?metabolite .

  ?gene rdfs:label ?gene_name .
  ?metabolite rdfs:label ?metabolite_name .

  # 3. THE BIOLOGICAL VALIDATION:
  # Check if BOTH the Gene AND the Metabolite participate in the EXACT SAME pathway
  ?gene biolink:participates_in ?pathway .
  ?metabolite biolink:participates_in ?pathway .

  ?pathway rdfs:label ?pathway_name .
}
ORDER BY ?pathway_name
Sending query to GraphDB (mtrKG)...
Success! Retrieved 6 rows.
Sa

,gene_name,metabolite_name,pathway_name
0,SLC22A5,acetylcarnitine (c2),Carnitine shuttle
1,SLC22A5,palmitoylcarnitine (c16),Carnitine shuttle
2,BBOX1,deoxycarnitine,Carnitine synthesis
3,ENPP6,choline phosphate,Glycerophospholipid catabolism
4,TMEM258,cholesterol,Maturation of DENV proteins
5,SLC22A3,carnitine,SLC-mediated transport of organic cations


c# Query 07

<font size="4">
We know this SNP causes Alzheimer's (from GWAS) and disrupts the Glutamine ratio (from rQTL). But does Glutamine actually exist in the brain where Alzheimer's happens?
</font>

In [7]:
df_results = query_graphdb("../analysis/007_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/007_query.rq
SPARQL query being executed:
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>

SELECT
  ?disease_name
  ?ratio_name
  ?numerator_name
  (GROUP_CONCAT(DISTINCT ?num_tissue; separator=" | ") AS ?numerator_tissues)
  ?denominator_name
  (GROUP_CONCAT(DISTINCT ?den_tissue; separator=" | ") AS ?denominator_tissues)
WHERE {
  # 1. Find the Clinical Trait / Disease
  ?gwas_assoc a biolink:Association ;
              biolink:has_subject ?snp ;
              biolink:has_object ?disease ;
              mtr:p_value ?gwas_p .

  ?disease a biolink:Disease ;
           rdfs:label ?disease_name .

  FILTER(?gwas_p < 0.05)

  # 2. Find the Metabolite Ratio linked to that SNP
  ?rqtl_assoc a biolink:Association ;
              biolink:has_subject ?snp ;
              biolink:has_object ?ratio .

  ?ratio a mtr:MetaboliteRatio ;
        

,disease_name,ratio_name,numerator_name,numerator_tissues,denominator_name,denominator_tissues
0,Alzheimer disease,lignoceroyl sphingomyelin (d18:1/24:0) to 1-st...,lignoceroyl sphingomyelin (d18:1/24:0),Blood | Cerebrospinal Fluid (CSF) | Saliva | U...,1-stearoyl-gpe (18:0),Feces | Placenta
1,Death in infancy,galactonate to indolepropionate,galactonate,Blood | Feces | Urine | Erythrocyte,indolepropionate,Blood | Feces | Saliva | Urine
2,HDL particle size,1-stearoyl-gpe (18:0) to glycerophosphorylchol...,1-stearoyl-gpe (18:0),Feces | Placenta,glycerophosphorylcholine (gpc),Blood | Breast Milk | Cerebrospinal Fluid (CSF...
3,VLDL particle size,"1-palmitoyl-2-linoleoyl-gpc (16:0/18:2) to 1,2...",1-palmitoyl-2-linoleoyl-gpc (16:0/18:2),Blood | Feces | Placenta | Saliva | Urine | Al...,"1,2-dipalmitoyl-gpc (16:0/16:0)",Blood | Feces | Placenta | Saliva | Urine
4,VLDL particle size,1-palmitoyl-2-linoleoyl-gpc (16:0/18:2) to 1-p...,1-palmitoyl-2-linoleoyl-gpc (16:0/18:2),Blood | Feces | Placenta | Saliva | Urine | Al...,1-palmitoyl-2-arachidonoyl-gpc (16:0/20:4n6),Blood | Feces | Placenta | Saliva | Urine | Al...
...,...,...,...,...,...,...
770,waist circumference,docosapentaenoate n3 DPA; 22:5n3 to tetradecad...,docosapentaenoate (n3 dpa; 22:5n3),Blood | Feces | Placenta | Urine,tetradecadienoate (14:2)*,
771,waist circumference,hexadecadienoate (16:2n6) to docosapentaenoate...,hexadecadienoate (16:2n6),Feces,docosapentaenoate (n6 dpa; 22:5n6),Adipose Tissue | Blood | Feces | Fibroblasts |...
772,waist circumference,linoleate (18:2n6) to docosapentaenoate (n6 dp...,linoleate (18:2n6),,docosapentaenoate (n6 dpa; 22:5n6),Adipose Tissue | Blood | Feces | Fibroblasts |...
773,waist circumference,tetradecadienoate (14:2)* to pimeloylcarnitine...,tetradecadienoate (14:2)*,,pimeloylcarnitine/3-methyladipoylcarnitine (c7...,Blood | Feces | Urine


# Query 09

<font size="4">
Our Knowledge Graph automatically deduced that [Disease] is linked to variant [SNP_ID], which causes an imbalance in the [Metabolite_Ratio]. This imbalance is causally driven by the [Causal_Gene], disrupting the [Biological_Pathway]
</font>

In [8]:
df_results = query_graphdb("../analysis/009_query.rq", repo_name="mtrKG")
#df_results.to_csv("../analysis/009_result.csv")
# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/009_query.rq
SPARQL query being executed:
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>

SELECT DISTINCT
  ?disease
  ?snp_id
  ?metabolite_ratio
  ?causal_gene
  ?biological_pathway
WHERE {
  # ---------------------------------------------------------
  # 1. CLINICAL: What complex disease is caused by a genetic variant?
  # ---------------------------------------------------------
  ?gwas_edge a biolink:Association ;
             biolink:has_subject ?snp ;
             biolink:has_object ?disease_node .

  ?disease_node a biolink:Disease ;
                rdfs:label ?disease .  # <--- FIXED THIS TO MATCH SELECT
FILTER(!CONTAINS(LCASE(?disease), "particle size"))
  # ---------------------------------------------------------
  # 2. METABOLIC: What metabolite ratio does that EXACT SAME variant disrupt?
  # --------------------------

,disease,snp_id,metabolite_ratio,causal_gene,biological_pathway
0,Alzheimer disease,rs390082,tricosanoyl sphingomyelin (d18:1/23:0)* to 1-s...,BEST1,Stimuli-sensing channels
1,Alzheimer disease,rs390082,tricosanoyl sphingomyelin (d18:1/23:0)* to 1-s...,BEST1,Ion channel transport
2,Alzheimer disease,rs390082,tricosanoyl sphingomyelin (d18:1/23:0)* to 1-s...,FADS1,Linoleic acid (LA) metabolism
3,Alzheimer disease,rs390082,tricosanoyl sphingomyelin (d18:1/23:0)* to 1-s...,FADS1,alpha-linolenic acid (ALA) metabolism
4,Alzheimer disease,rs390082,tricosanoyl sphingomyelin (d18:1/23:0)* to 1-s...,FADS1,alpha-linolenic (omega3) and linoleic (omega6)...
...,...,...,...,...,...
35719,whole body water mass,rs3961283,"n6,n6,n6-trimethyllysine to argininate*",TRADD,Dimerization of procaspase-8
35720,whole body water mass,rs3961283,"n6,n6,n6-trimethyllysine to deoxycarnitine",TRADD,Defective RIPK1-mediated regulated necrosis
35721,whole body water mass,rs3961283,"n6,n6,n6-trimethyllysine to argininate*",TRADD,Defective RIPK1-mediated regulated necrosis
35722,whole body water mass,rs3961283,"n6,n6,n6-trimethyllysine to deoxycarnitine",TRADD,Caspase-8 and -10 mediated induction of NF-kB


# Query 10

<font size="4">
If a genetic variant is known to cause a complex disease, what is the underlying metabolic mechanism, which gene is driving it, and are there any existing drugs that target that gene?"
</font>

In [18]:
df_results = query_graphdb("../analysis/010_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/010_query.rq
SPARQL query being executed:
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>

SELECT
  ?disease
  ?snp_id
  ?metabolite_ratio
  ?causal_gene
  ?biological_pathway
  ?p_value
  ?beta
  (GROUP_CONCAT(DISTINCT ?drug_name; separator=", ") AS ?targeting_drugs)
WHERE {
  # ---------------------------------------------------------
  # 1. CLINICAL: What complex disease is caused by a genetic variant?
  # ---------------------------------------------------------
  ?gwas_edge a biolink:Association ;
             biolink:has_subject ?snp ;
             biolink:has_object ?disease_node .

  ?disease_node a biolink:Disease ;
                rdfs:label ?disease .

  FILTER(!CONTAINS(LCASE(?disease), "particle size"))

  # ---------------------------------------------------------
  # 2. METABOLIC: What metabolite ratio does that varia

,disease,snp_id,metabolite_ratio,causal_gene,biological_pathway,p_value,beta,targeting_drugs
0,Alzheimer disease,rs4147929,1-(1-enyl-palmitoyl)-2-oleoyl-gpc (p-16:0/18:1...,GRIN3B,Transmission across Chemical Synapses,2.34924072256025e-19,-0.318519973265192,"FELBAMATE, ORPHENADRINE CITRATE, AMANTADINE HY..."
1,Alzheimer disease,rs4147929,1-(1-enyl-palmitoyl)-2-palmitoyl-gpc (p-16:0/1...,GRIN3B,Transmission across Chemical Synapses,9.47566535232506e-17,-0.293734779326072,"FELBAMATE, ORPHENADRINE CITRATE, AMANTADINE HY..."
2,Alzheimer disease,rs4147929,1-(1-enyl-palmitoyl)-2-oleoyl-gpc (p-16:0/18:1...,GRIN3B,Neuronal System,2.34924072256025e-19,-0.318519973265192,"FELBAMATE, ORPHENADRINE CITRATE, AMANTADINE HY..."
3,Alzheimer disease,rs4147929,1-(1-enyl-palmitoyl)-2-palmitoyl-gpc (p-16:0/1...,GRIN3B,Neuronal System,9.47566535232506e-17,-0.293734779326072,"FELBAMATE, ORPHENADRINE CITRATE, AMANTADINE HY..."
4,Alzheimer disease,rs4147929,1-(1-enyl-palmitoyl)-2-oleoyl-gpc (p-16:0/18:1...,GRIN3B,Neurotransmitter receptors and postsynaptic si...,2.34924072256025e-19,-0.318519973265192,"FELBAMATE, ORPHENADRINE CITRATE, AMANTADINE HY..."
...,...,...,...,...,...,...,...,...
3585,waist circumference,rs2900478,"androstenediol (3beta,17beta) monosulfate (2) ...",CYP3A7,Xenobiotics,2.30003453836274e-35,-0.376301656008259,"RITONAVIR, COBICISTAT"
3586,waist circumference,rs2900478,"androstenediol (3beta,17beta) monosulfate (2) ...",CYP3A7,Cytochrome P450 - arranged by substrate type,2.30003453836274e-35,-0.376301656008259,"RITONAVIR, COBICISTAT"
3587,waist circumference,rs2900478,"androstenediol (3beta,17beta) monosulfate (2) ...",CYP3A7,Phase I - Functionalization of compounds,2.30003453836274e-35,-0.376301656008259,"RITONAVIR, COBICISTAT"
3588,waist circumference,rs2900478,"androstenediol (3beta,17beta) monosulfate (2) ...",CYP3A7,Biological oxidations,2.30003453836274e-35,-0.376301656008259,"RITONAVIR, COBICISTAT"


<font size="4">
While the previous SELECT query functions as a semantic retrieval tool, this CONSTRUCT query acts as an active symbolic inference engine. It is designed to logically synthesize entirely new therapeutic hypotheses based on the multi-omics topology of the graph, effectively automating the discovery of precision drug repurposing candidates.

The Inference Logic (The Premise):
The query evaluates a strict, 5-hop mechanistic premise:

IF a genetic variant is associated with a complex disease (via GWAS)...

AND that same variant disrupts a specific metabolite ratio (via rQTL)...

AND a gene has a mathematically proven causal effect on that ratio (via Mendelian Randomization)...

AND that gene participates in a known biological pathway...

AND an approved drug is known to target or interact with that gene...

THEN: The drug is computationally predicted to target the downstream complex disease.

Output & Provenance Tracking (Reification):
To prevent the epistemological conflation of computational hypotheses with biologically validated ground-truth, this query does not generate definitive clinical edges (e.g., biolink:treats). Instead, it executes two highly controlled graph operations:

The Direct Edge: It instantiates a cautious mtr:predicted_to_target edge between the drug and the disease for simplified downstream visualization.

The Reified Provenance Node: It constructs a complex mtr:InferredTherapeuticAssociation node. This this allows us to distinguish between the inferred relationships and the relationships we obtained from the datasets
</font>

In [21]:
df_results = query_graphdb("../analysis/010_query_construct.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/010_query_construct.rq
SPARQL query being executed:
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>
PREFIX prov: <http://www.w3.org/ns/prov#>

CONSTRUCT {
  # 1. The Cautious Direct Edge (For simple graph visualization)
  ?drug mtr:predicted_to_target ?disease_node .

  # 2. The Reified Association Node (For rigorous provenance)
  ?inference_node a mtr:InferredTherapeuticAssociation ;
                  biolink:subject ?drug ;
                  biolink:object ?disease_node ;
                  biolink:predicate mtr:predicted_to_target ;

                  # EXPLICITLY state this is a graph prediction, not clinical truth
                  prov:wasGeneratedBy mtr:SymbolicGraphInference ;
                  biolink:has_evidence mtr:ComputationalGraphTraversal ;
                  rdfs:comment "This edge was logically inferred from structur

D:\#University\#MS-SystemsBio\Period_04\Building and Mining Knowledge Graphs\mtrKG\.venv\Lib\site-packages\SPARQLWrapper\Wrapper.py:1179: RuntimeWarning: Format requested was JSON, but N3 (application/n-triples;charset=UTF-8) has been returned by the endpoint
  warnings.warn(


GraphDB Connection/Query Error: a bytes-like object is required, not 'str'
-> Make sure GraphDB is running and your repository name is correct!


None

<font size="4">

This query extracts the foundational architecture of your project. It finds a specific region of the genome (SNP), identifies the causal gene at that locus, and shows exactly which metabolic ratio it disrupts, filtered by strict mathematical confidence (p-value).
</font>

# Query 11

In [17]:
df_results = query_graphdb("../analysis/011_query.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/011_query.rq
SPARQL query being executed:
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX biolink: <https://w3id.org/biolink/vocab/>
PREFIX mtr: <https://metabolite-ratio-app.unil.ch/rqtl/>


SELECT ?snp ?causalGeneLabel ?ratioLabel ?p_value ?dataset
WHERE {
    # 1. Find the Causal Assessment Node
    ?assessment a mtr:CausalAssessment ;
                mtr:associated_variant ?snp ;
                mtr:has_exposure ?geneNode ;
                mtr:has_outcome ?ratioNode ;
                mtr:p_ivw ?p_value ;
                mtr:dataset_source ?dataset .

    # 2. Extract Human-Readable Labels
    ?geneNode rdfs:label ?causalGeneLabel .
    ?ratioNode rdfs:label ?ratioLabel .
    # 3. Filter for high-confidence Mendelian Randomization results
    FILTER(?p_value < 0.05)
}
ORDER BY ?p_value
LIMIT 10
Sending query to GraphDB (mtrKG)...
Success! Retrieved 10 rows.
Saved results to CSV: ../analysis/011_query.csv


,snp,causalGeneLabel,ratioLabel,p_value,dataset
0,http://identifiers.org/dbsnp/rs7573275,ALMS1 RNA,2-aminooctanoate to n-acetyl-2-aminooctanoate*,0e+00,GTEx
1,http://identifiers.org/dbsnp/rs6752270,ALMS1 RNA,"n2,n5-diacetylornithine to n-delta-acetylornit...",0e+00,GTEx
2,http://identifiers.org/dbsnp/rs7572565,ALMS1 RNA,"n2,n5-diacetylornithine to n-delta-acetylornit...",0e+00,GTEx
3,http://identifiers.org/dbsnp/rs7573275,ALMS1 RNA,methionine sulfone to n-acetylglutamine,0e+00,GTEx
4,http://identifiers.org/dbsnp/rs7573275,ALMS1 RNA,methionine sulfone to n-acetyl-1-methylhistidine*,0e+00,GTEx
5,http://identifiers.org/dbsnp/rs7573275,ALMS1 RNA,n-acetyl-2-aminooctanoate* to n-delta-acetylor...,0e+00,GTEx
6,http://identifiers.org/dbsnp/rs7573275,ALMS1 RNA,n-acetylleucine to n-delta-acetylornithine,0e+00,GTEx
7,http://identifiers.org/dbsnp/rs7573275,ALMS1 RNA,n-acetyltyrosine to n-delta-acetylornithine,0e+00,GTEx
8,http://identifiers.org/dbsnp/rs58242989,ALMS1 RNA,n-acetylglutamine to n-delta-acetylornithine,0e+00,GTEx
9,http://identifiers.org/dbsnp/rs62149622,ALMS1 RNA,n-acetylglutamine to n-delta-acetylornithine,0e+00,GTEx


# Query 13
Counting nodes and triples

In [23]:
df_results = query_graphdb("../analysis/013_query_count_triples.rq", repo_name="mtrKG")

# Display the results
display(df_results)

Reading SPARQL query from file: ../analysis/013_query_count_triples.rq
SPARQL query being executed:
SELECT
    (COUNT(*) AS ?triple_count)
    (COUNT(DISTINCT ?s) AS ?node_count)
    WHERE {
    ?s ?p ?o .
}
Sending query to GraphDB (mtrKG)...
Success! Retrieved 1 rows.
Saved results to CSV: ../analysis/013_query_count_triples.csv


,triple_count,node_count
0,9250148,379870
